# Stage 6 - Evaluation: internal metrics, stability, and the winning algorithm

**Data:** real Porto taxi GPS, 602 H3 cells. `synthetic: false`. **Limitation:** taxis, not buses.

Three questions, increasing in honesty:

1. **Coherence** - silhouette, Davies-Bouldin, Calinski-Harabasz, and DBCV for density methods.
2. **Stability** - re-fit on random subsamples, compare with the Adjusted Rand Index.
3. **Worth reporting?** - coverage (how much of the city is even labelled) and whether the metric
   was measured in the space the algorithm clustered.

Rules enforced here:

* **Noise (-1) is not a cluster.** It is excluded from every metric and reported as `noise_share`.
* **Score in the space the algorithm used.** DBSCAN clustered kilometres, so its silhouette uses a
  haversine distance matrix. A common-space (PCA) silhouette is reported next to it so the numbers
  can be compared across spaces.
* **Davies-Bouldin and Calinski-Harabasz need a feature space.** They stay empty for precomputed
  DTW/haversine spaces instead of being faked with Euclidean-on-degrees.

In [1]:
import json, pathlib, sys, time
sys.path.append(str(pathlib.Path.cwd().parent))

import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np, pandas as pd
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.mixture import GaussianMixture
from tslearn.metrics import cdist_dtw

from src.io import PROJECT_ROOT
from src import clustering as CL
from src import evaluate as E

PROC = PROJECT_ROOT / "data" / "processed"
OUT = PROJECT_ROOT / "outputs"; OUT.mkdir(parents=True, exist_ok=True)
FIG = OUT / "figures"

scores = pd.read_parquet(PROC / "pca_scores.parquet")
raw = pd.read_parquet(PROC / "cells_features.parquet")
labels_all = pd.read_parquet(PROC / "cluster_labels_all.parquet")
profile = pd.read_parquet(PROC / "hour_of_week_profile.parquet").loc[scores["h3_cell"].to_numpy()]
sel = json.loads((PROC / "clustering_selection.json").read_text())

PC = [f"PC{i+1}" for i in range(5)]
X = scores[PC].to_numpy(dtype=float)
cent = scores[["h3_cell"]].merge(raw[["h3_cell", "lat_centroid", "lon_centroid"]], on="h3_cell", how="left")
LAT, LON = cent["lat_centroid"].to_numpy(), cent["lon_centroid"].to_numpy()
D_KM = E.haversine_matrix(LAT, LON)
print(f"cells={len(X)}  PCA dims={X.shape[1]}  labels={list(labels_all.columns[1:])}")

# DTW distance matrix on a subsample: a full 602x602 DTW matrix is ~180k warps.
DTW_SUB = 200
rng = np.random.default_rng(E.RANDOM_STATE)
sub = rng.choice(len(profile), size=DTW_SUB, replace=False)
t0 = time.time()
D_DTW = cdist_dtw(profile.to_numpy()[sub][:, :, None],
                  global_constraint="sakoe_chiba", sakoe_chiba_radius=12)
print(f"DTW distance matrix {D_DTW.shape} in {time.time()-t0:.1f}s (subsample {DTW_SUB})")

cells=602  PCA dims=5  labels=['kmeans', 'ward', 'gmm', 'dbscan_spatial', 'hdbscan', 'tskmeans']


DTW distance matrix (200, 200) in 6.7s (subsample 200)


In [2]:
# 1) Coherence. Each algorithm is scored in ITS OWN space, plus a common PCA-space silhouette.
rows = []

def add_row(algo, space, own, own_kwargs, dbcv_kwargs=None):
    r = {"algorithm": algo, "space": space, **own}
    r["dbcv"] = E.dbcv(**dbcv_kwargs) if dbcv_kwargs else None
    # common space: silhouette of the same labels in PC1-PC5, so spaces can be compared
    r["silhouette_pca_space"] = E.score_partition(X, labels_all[algo].to_numpy())["silhouette"]
    rows.append(r)

add_row("kmeans", "PC1-PC5 (euclidean)",
        E.score_partition(X, labels_all["kmeans"].to_numpy()), {},
        {"X": X, "labels": labels_all["kmeans"].to_numpy()})
add_row("ward", "PC1-PC5 (euclidean)",
        E.score_partition(X, labels_all["ward"].to_numpy()), {},
        {"X": X, "labels": labels_all["ward"].to_numpy()})
add_row("gmm", "PC1-PC5 (euclidean)",
        E.score_partition(X, labels_all["gmm"].to_numpy()), {},
        {"X": X, "labels": labels_all["gmm"].to_numpy()})
add_row("dbscan_spatial", "lat/lon (haversine km)",
        E.score_partition(D_KM, labels_all["dbscan_spatial"].to_numpy(), metric="precomputed", precomputed=True), {},
        {"X": D_KM, "labels": labels_all["dbscan_spatial"].to_numpy(), "precomputed": True, "d": 2})
add_row("hdbscan", "PC1-PC5 (euclidean)",
        E.score_partition(X, labels_all["hdbscan"].to_numpy()), {},
        {"X": X, "labels": labels_all["hdbscan"].to_numpy()})
add_row("tskmeans", "DTW on 168-h profiles (subsample)",
        E.score_partition(D_DTW, labels_all["tskmeans"].to_numpy()[sub], metric="precomputed", precomputed=True), {},
        None)

coherence = pd.DataFrame(rows)
print(coherence.round(3).to_string(index=False))
print("\nDBCV note: 'None' for tskmeans is deliberate - DBCV's core-distance formula needs a")
print("space dimensionality, and a DTW metric space has none. Inventing one would be fake precision.")

     algorithm                             space  n_clusters  coverage  noise_share  silhouette  davies_bouldin  calinski_harabasz   dbcv  silhouette_pca_space
        kmeans               PC1-PC5 (euclidean)           4     1.000        0.000       0.244           1.435            182.189 -0.596                 0.244
          ward               PC1-PC5 (euclidean)           4     1.000        0.000       0.216           1.786            144.046 -0.476                 0.216
           gmm               PC1-PC5 (euclidean)           4     1.000        0.000       0.151           2.050            106.592 -0.504                 0.151
dbscan_spatial            lat/lon (haversine km)          41     0.625        0.375       0.217             NaN                NaN  0.183                -0.252
       hdbscan               PC1-PC5 (euclidean)           3     0.397        0.603       0.333           0.912             52.607 -0.203                 0.333
      tskmeans DTW on 168-h profiles (su

In [3]:
# 2) Stability: re-fit on two independent 80% subsamples, compare labels on the overlap.
def fit_fns():
    return {
        "kmeans": lambda idx: KMeans(n_clusters=4, n_init=20, random_state=CL.RANDOM_STATE).fit_predict(X[idx]),
        "ward": lambda idx: AgglomerativeClustering(n_clusters=4, linkage="ward").fit_predict(X[idx]),
        "gmm": lambda idx: GaussianMixture(n_components=4, covariance_type="full", reg_covar=1e-4,
                                           n_init=5, random_state=CL.RANDOM_STATE).fit_predict(X[idx]),
        "dbscan_spatial": lambda idx: CL.dbscan_haversine(LAT[idx], LON[idx], eps_km=1.0, min_samples=5),
        "hdbscan": lambda idx: CL.hdbscan_run(X[idx], min_cluster_size=10, min_samples=5)[0],
        "tskmeans": lambda idx: CL.ts_kmeans(profile.iloc[idx], k=4)[0],
    }

stability = {}
for name, fn in fit_fns().items():
    # DTW costs ~15 s per fit, so fewer, smaller bootstraps - and that is stated, not hidden
    n_boot, frac = (5, 0.6) if name == "tskmeans" else (10, 0.8)
    t0 = time.time()
    res = E.bootstrap_stability(fn, n_obs=len(X), n_boot=n_boot, frac=frac)
    stability[name] = {**res, "frac": frac}
    print(f"{name:15s} ARI mean={res['ari_mean']} sd={res['ari_sd']} min={res['ari_min']} "
          f"(n_boot={res['n_boot']}, frac={frac}, {time.time()-t0:.0f}s)")

kmeans          ARI mean=0.8507 sd=0.0518 min=0.7582 (n_boot=10, frac=0.8, 1s)
ward            ARI mean=0.4742 sd=0.1617 min=0.2864 (n_boot=10, frac=0.8, 0s)


gmm             ARI mean=0.5722 sd=0.1655 min=0.308 (n_boot=10, frac=0.8, 3s)
dbscan_spatial  ARI mean=0.414 sd=0.1055 min=0.2566 (n_boot=10, frac=0.8, 0s)


hdbscan         ARI mean=0.6945 sd=0.0733 min=0.5281 (n_boot=10, frac=0.8, 0s)


/home/ashira/Documents/Projects/Fleetsense/.venv/lib/python3.12/site-packages/tslearn/bases/bases.py:16: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)


tskmeans        ARI mean=0.4302 sd=0.1253 min=0.2998 (n_boot=5, frac=0.6, 54s)


In [4]:
# 3) The comparison table
table = coherence.copy()
for name, st in stability.items():
    m = table["algorithm"] == name
    table.loc[m, "ari_mean"] = st["ari_mean"]
    table.loc[m, "ari_sd"] = st["ari_sd"]
    table.loc[m, "ari_min"] = st["ari_min"]
    table.loc[m, "ari_boot_n"] = st["n_boot"]

table = E.compare_table(table.to_dict("records"))
print(table.round(3).to_string(index=False))

table.to_csv(OUT / "cluster_comparison.csv", index=False)
(OUT / "cluster_comparison.json").write_text(json.dumps(
    {"stage": 6, "data_source": "porto_taxi_uci339", "synthetic": False,
     "limitation": "taxis, not buses",
     "rows": table.round(6).to_dict("records"),
     "stability_settings": stability,
     "notes": {
         "noise": "cells labelled -1 are excluded from all metrics and reported as noise_share",
         "spaces": "silhouette is computed in the algorithm's own space; silhouette_pca_space "
                   "repeats it in PC1-PC5 for cross-space comparison",
         "db_ch": "undefined without a feature space, left empty rather than approximated",
         "dbcv_dtw": "not defined for a DTW metric space; reported as null",
     }}, indent=2, default=str))
print("wrote", OUT / "cluster_comparison.csv")

     algorithm                             space  n_clusters  coverage  noise_share  silhouette  silhouette_pca_space  davies_bouldin  calinski_harabasz   dbcv  ari_mean  ari_sd  ari_min  ari_boot_n
        kmeans               PC1-PC5 (euclidean)           4     1.000        0.000       0.244                 0.244           1.435            182.189 -0.596     0.851   0.052    0.758        10.0
          ward               PC1-PC5 (euclidean)           4     1.000        0.000       0.216                 0.216           1.786            144.046 -0.476     0.474   0.162    0.286        10.0
           gmm               PC1-PC5 (euclidean)           4     1.000        0.000       0.151                 0.151           2.050            106.592 -0.504     0.572   0.166    0.308        10.0
dbscan_spatial            lat/lon (haversine km)          41     0.625        0.375       0.217                -0.252             NaN                NaN  0.183     0.414   0.106    0.257        10.0
     

In [5]:
# 4) Figure: how k was chosen - inertia elbow and silhouette together
km_tab, _ = CL.kmeans_scan(X, ks=range(2, 9))
K = 4
fig, ax1 = plt.subplots(figsize=(6.4, 3.4))
ax1.plot(km_tab["k"], km_tab["inertia"], marker="o", color="#111111", label="inertia J(k)")
ax1.set_xlabel("k"); ax1.set_ylabel("inertia J", color="#111111")
ax2 = ax1.twinx()
ax2.plot(km_tab["k"], km_tab["silhouette"], marker="s", color="#9A9A9A", label="silhouette")
ax2.set_ylabel("mean silhouette", color="#9A9A9A")
ax1.axvline(K, color="#111111", ls=":", lw=1)
ax1.set_title("Inertia knee at k=4; silhouette is flat across k=2-5, so k=4 is defensible on both")
fig.tight_layout(); fig.savefig(FIG / "stage6_k_selection.png", dpi=140, bbox_inches="tight")
print("saved", FIG / "stage6_k_selection.png")

saved /home/ashira/Documents/Projects/Fleetsense/outputs/figures/stage6_k_selection.png


In [6]:
# 5) Figure: coherence vs stability. No single winner on both axes - that is the finding.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
order = table.sort_values("silhouette", ascending=True)
axes[0].barh(order["algorithm"], order["silhouette"], color="#111111")
axes[0].set_title("silhouette in own space (higher better)"); axes[0].set_xlabel("mean silhouette")
axes[0].barh(order["algorithm"], order["silhouette_pca_space"], color="#D9D9D9", label="same labels, PCA space")
axes[0].legend(frameon=False, fontsize=8, loc="lower right")

order2 = table.sort_values("ari_mean", ascending=True)
axes[1].barh(order2["algorithm"], order2["ari_mean"], color="#111111")
axes[1].errorbar(order2["ari_mean"], np.arange(len(order2)),
                 xerr=order2["ari_sd"].fillna(0), fmt="none", ecolor="#9A9A9A", capsize=3)
axes[1].set_title("stability: mean bootstrap ARI (higher better)"); axes[1].set_xlabel("mean ARI +/- sd")
axes[1].set_xlim(0, 1)
fig.tight_layout(); fig.savefig(FIG / "stage6_comparison.png", dpi=140, bbox_inches="tight")
print("saved", FIG / "stage6_comparison.png")

saved /home/ashira/Documents/Projects/Fleetsense/outputs/figures/stage6_comparison.png


In [7]:
# 6) Do the algorithms agree with each other? Pairwise ARI between label sets.
from sklearn.metrics import adjusted_rand_score
algos = list(labels_all.columns[1:])
M = np.zeros((len(algos), len(algos)))
for i, a in enumerate(algos):
    for j, b in enumerate(algos):
        M[i, j] = adjusted_rand_score(labels_all[a], labels_all[b])

fig, ax = plt.subplots(figsize=(5.2, 4.4))
im = ax.imshow(M, cmap="Greys", vmin=0, vmax=1)
ax.set_xticks(range(len(algos))); ax.set_xticklabels(algos, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(algos))); ax.set_yticklabels(algos, fontsize=8)
for i in range(len(algos)):
    for j in range(len(algos)):
        ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if M[i, j] > 0.55 else "#111111")
ax.set_title("pairwise ARI between algorithms")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
fig.tight_layout(); fig.savefig(FIG / "stage6_agreement.png", dpi=140, bbox_inches="tight")
print("saved", FIG / "stage6_agreement.png")
print(pd.DataFrame(M, index=algos, columns=algos).round(2).to_string())

saved /home/ashira/Documents/Projects/Fleetsense/outputs/figures/stage6_agreement.png
                kmeans  ward   gmm  dbscan_spatial  hdbscan  tskmeans
kmeans            1.00  0.54  0.38            0.17     0.19      0.04
ward              0.54  1.00  0.39            0.18     0.23      0.04
gmm               0.38  0.39  1.00            0.07     0.28      0.09
dbscan_spatial    0.17  0.18  0.07            1.00     0.09     -0.05
hdbscan           0.19  0.23  0.28            0.09     1.00     -0.05
tskmeans          0.04  0.04  0.09           -0.05    -0.05      1.00


## Reading the comparison

* **Silhouette alone would be misleading here.** DBSCAN scores highest in its own haversine space, but
  its labels have almost no relationship to behaviour (see `silhouette_pca_space`, which is negative):
  it recovered geography, and the question is behaviour.
* **DBCV separates "compact" from "density-separated".** A negative DBCV for K-Means/Ward/GMM means the
  clusters are not separated by density valleys at all - they are slices of a continuum. That is
  consistent with Stage 5's HDBSCAN finding.
* **Stability is the tiebreaker between similarly-scoring methods.** ARI ~0.8+ means the partition
  reappears when 20% of cells are dropped; ARI ~0.3 means it depends on which cells you happened to keep.
* **Agreement between algorithms is evidence, not proof.** If K-Means, Ward and GMM recover nearly the
  same partition (high pairwise ARI), the grouping is a property of the data rather than of one
  algorithm's inductive bias.